# Tutorial: Layer 2 KBO Foreign Archetype Mining


## 1. 목적

            2번 레이어는 **KBO 외국인 선수 성공/실패 유형 마이닝**이다.

            여기서의 질문은 이렇다.

            > 과거 KBO 외국인 선수 중 어떤 사전 프로필이 성공/실패와 반복적으로 연결됐는가?

            ## 사용한 모델/방법

            - historical label mart: 과거 외국인 선수 시즌을 성공/실패/교체/재계약 관점으로 라벨링
            - archetype clustering/profile: 입단 전 특성으로 유형을 나눔
            - association rule lift: 특정 특성 조합이 성공/실패율을 얼마나 올리는지 계산
            - permutation p-value: 작은 표본에서 우연히 나온 규칙인지 점검
            - backfill coverage audit: 모델이 배울 수 있는 과거 데이터가 충분한지 확인


## 2. 중요한 태도

            이 레이어는 "이 유형이면 무조건 성공"을 말하지 않는다.

            작은 표본의 외국인 선수 시장에서는 오히려 다음 문장이 더 중요하다.

            > 규칙은 후보 점수를 바로 만들기보다, 어떤 질문을 스카우팅 카드에 반드시 넣어야 하는지를 정한다.


In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent

OUT = ROOT / "outputs" / "tables"

def read_table(filename):
    path = OUT / filename
    if not path.exists():
        print(f"[missing] {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

def take_cols(df, cols, n=10):
    keep = [c for c in cols if c in df.columns]
    if not keep:
        return df.head(n)
    return df.loc[:, keep].head(n)

def count_by(df, cols):
    keep = [c for c in cols if c in df.columns]
    if not keep or df.empty:
        return pd.DataFrame()
    return df.groupby(keep, dropna=False).size().reset_index(name="rows").sort_values("rows", ascending=False)

def assert_candidate_names_locked(df):
    sensitive_cols = {"player_name", "search_name", "team_or_org", "player_id"}
    exposed = sensitive_cols.intersection(df.columns)
    if exposed:
        print(f"Candidate-sensitive columns exist but are not displayed here: {sorted(exposed)}")


In [ ]:
gate = read_table("recruitment_gate_status_v33.csv")
take_cols(gate[gate["gate"].eq("G2")], ["gate", "layer", "progress_pct", "status", "decision", "blocking_gap"])


In [ ]:
blueprint = read_table("modeling_blueprint_registry_v1.csv")
layer2_models = blueprint[
    blueprint["layer"].astype(str).str.contains("archetype|translation|failure", case=False, regex=True, na=False)
]
take_cols(layer2_models, ["model_id", "layer", "model_family", "target_or_score", "main_features", "validation", "promotion_rule"], n=12)


## 3. 과거 선수 유형 프로필

            아래 표는 선수 이름이 아니라 유형 단위 요약이다.

            핵심 컬럼:

            - rows: 해당 유형에 속한 과거 사례 수
            - success_rate/failure_rate: KBO 입성 후 결과
            - prearrival_*_fingerprint: 입단 전 강점/위험 특성 요약


In [ ]:
profiles = read_table("kbo_foreign_archetype_prearrival_profile_v0_2.csv")
take_cols(
    profiles,
    ["role_model_family", "archetype_cluster_id", "archetype_name", "rows", "model_ready_rows", "success_rate", "failure_rate", "prearrival_strength_fingerprint", "prearrival_risk_fingerprint", "prearrival_profile_gate"],
    n=10,
)


## 4. 규칙 기반 lift 확인

            lift는 "전체 평균보다 특정 조건 안에서 목표 비율이 얼마나 달라졌는가"를 본다.

            예시:

            - target이 failure라면 lift가 높을수록 위험 신호다.
            - support_rows가 너무 작으면 우연일 수 있으므로 바로 후보 점수로 쓰지 않는다.
            - rule_gate가 research_only이면 발표에서는 질문/검증 포인트로만 사용한다.


In [ ]:
rules = read_table("kbo_foreign_archetype_rule_lifts_v0_2.csv")
rules_view = rules.sort_values(["rule_gate", "abs_rate_delta"], ascending=[True, False])
take_cols(
    rules_view,
    ["role_model_family", "signal_type", "rule", "target", "support_rows", "target_rate_inside_rule", "role_base_target_rate", "rate_delta_vs_role_base", "lift_vs_role_base", "permutation_p_value", "rule_gate", "release_policy"],
    n=12,
)


In [ ]:
coverage = read_table("layer2_backfill_coverage_recalibration_v0_1.csv")
take_cols(coverage, coverage.columns.tolist(), n=10)


## 5. 발표용 한 줄

            2번 레이어는 과거 외국인 선수 사례를 사용해 "KBO에서 먹히는 유형과 위험한 유형"을 찾는 단계다. 하지만 표본이 작기 때문에, 복잡한 규칙은 최종 점수보다 스카우팅 질문으로 사용한다.

            ## 연습문제

            rule_lifts 표에서 support_rows가 충분하고 permutation_p_value가 낮은 규칙을 하나 고른 뒤, 그 규칙을 후보 검토 질문으로 바꿔보자.


In [ ]:
candidate_rules = rules[
    (rules["support_rows"] >= 5)
    & (rules["permutation_p_value"] <= 0.20)
].copy()
take_cols(
    candidate_rules.sort_values("abs_rate_delta", ascending=False),
    ["role_model_family", "rule", "target", "support_rows", "rate_delta_vs_role_base", "permutation_p_value", "semantic_alignment", "rule_gate"],
    n=8,
)
